# Whale / Boat / Water Chip Classifier — Training Notebook

Trains three models to give each 500x500 SkySat chip one label: **water**, **whale**, or **boat**.

| Model | Taken from |
| --- | --- |
| EfficientNet-B0 (CNN) | ICS 605 AI-image detector, `notebooks/train_detection_local.ipynb` |
| DeiT-Tiny (ViT) | same notebook |
| SatViT-V2 encoder + classification head | Joe's flood project, `satvit/ICS605_final_SatViT_model_train/` |

The training loop, early stopping and checkpointing are the ones from the 605 notebook. The SatViT
encoder, weight loading, partial freezing and per-band normalization are Joe's. What's new is the
data loading for the 4-band GeoTIFFs and a classification head for SatViT in place of the flood
decoder.

**Data:** `data/whale/data/` from `notebooks/ingestion/data_ingester.ipynb`.

**SatViT weights:** download `SatViT_V2.pt` (link in `satvit/README.md`) into `checkpoints/`.
Without it the SatViT run is skipped and the other two still train.

With 144 chips, treat these scores as a smoke test, not a result. The test split has about 22 chips.

## 1. Setup

In [ ]:
import sys

if 'google.colab' in sys.modules:
    !pip install -q rasterio einops timm

In [ ]:
import copy
import gc
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    f1_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

print(f'Python:  {sys.version.split()[0]}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: no CUDA detected. Training will be very slow on CPU.')

## 2. Config

In [ ]:
ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()),
    Path.cwd(),
)
DATA_DIR = ROOT / 'data' / 'whale' / 'data'
CKPT_DIR = ROOT / 'checkpoints'
SATVIT_DIR = ROOT / 'satvit' / 'ICS605_final_SatViT_model_train'
SATVIT_WEIGHTS = CKPT_DIR / 'SatViT_V2.pt'

CLASSES = ['water', 'whale', 'boat']
RGB_BANDS = [2, 1, 0]  # the chips store Blue, Green, Red, NIR

MODELS = {
    'efficientnet_b0': {'timm_name': 'efficientnet_b0', 'img_size': 224, 'batch_size': 16},
    'deit_tiny': {'timm_name': 'deit_tiny_patch16_224', 'img_size': 224, 'batch_size': 16},
    # SatViT-V2 takes 256x256 inputs in 8x8 patches (1024 patches); batch 6 is Joe's setting.
    'satvit': {'img_size': 256, 'batch_size': 6},
}

NUM_EPOCHS = 30
LR = 1e-4
PATIENCE = 8
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
NUM_WORKERS = 2
SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
CKPT_DIR.mkdir(exist_ok=True)
print(f'Device: {DEVICE}')
print(f'Data:   {DATA_DIR}')

## 3. Load chips and labels

A chip's label comes from which label folder has a file with its name. Chips in neither folder are
water. No chip currently has both a whale and a boat mask; if one did, it would count as whale.
All 144 chips fit in memory (~290 MB as uint16), so they're read once here.

In [ ]:
def label_for(name):
    if (DATA_DIR / 'labels' / 'whales' / name).exists():
        return CLASSES.index('whale')
    if (DATA_DIR / 'labels' / 'boats' / name).exists():
        return CLASSES.index('boat')
    return CLASSES.index('water')


names = sorted(p.name for p in (DATA_DIR / 'images').glob('*.tif'))
images = []
for name in names:
    with rasterio.open(DATA_DIR / 'images' / name) as src:
        images.append(src.read())  # (4, 500, 500) uint16
labels = np.array([label_for(n) for n in names])

print(f'{len(names)} chips')
for i, c in enumerate(CLASSES):
    print(f'  {c:6s} {np.sum(labels == i)}')

## 4. Train / val / test split

Stratified, so each split keeps the class ratios (that matters with only 8 water chips). The chips
have no scene IDs, so chips from the same scene can land in different splits, which can make the
scores look better than they really are.

In [ ]:
all_idx = np.arange(len(names))
trainval_idx, test_idx = train_test_split(
    all_idx, test_size=TEST_SPLIT, stratify=labels, random_state=SEED
)
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=VAL_SPLIT / (1 - TEST_SPLIT),
    stratify=labels[trainval_idx],
    random_state=SEED,
)

for split, idx in [('train', train_idx), ('val', val_idx), ('test', test_idx)]:
    counts = '  '.join(f'{c}={np.sum(labels[idx] == i)}' for i, c in enumerate(CLASSES))
    print(f'{split:5s} {len(idx):3d}   {counts}')

## 5. Dataset

Each model family gets the input it was pretrained on:

- **EfficientNet / DeiT:** RGB only. Each band gets one fixed 2nd–98th percentile stretch, taken
  from the training chips, then the ImageNet normalization the 605 notebook used.
- **SatViT:** Joe's `_normalize_image`, which z-scores each band of each chip separately. SatViT-V2
  expects 15 bands (2 Sentinel-1, then Sentinel-2 B1–B12 with B8A). Our Blue, Green, Red and NIR go
  where Sentinel-2 B2, B3, B4 and B8 sit, and the other 11 bands are left at 0.

Chips are resized from 500 to the model's input size. Training adds random flips (as in Joe's loop)
and 90° rotations, since overhead imagery has no "up".

In [ ]:
# Fixed per-band stretch from training chips (every 4th pixel is plenty for percentiles).
train_pixels = np.stack([images[i][:, ::4, ::4] for i in train_idx]).astype(np.float32)
BAND_LO = torch.tensor(np.percentile(train_pixels, 2, axis=(0, 2, 3))).view(4, 1, 1)
BAND_HI = torch.tensor(np.percentile(train_pixels, 98, axis=(0, 2, 3))).view(4, 1, 1)
del train_pixels

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

SATVIT_CHANNELS = 15
SATVIT_BAND_SLOTS = [3, 4, 5, 9]  # S2 B2, B3, B4, B8 after the 2 Sentinel-1 bands


class WhaleChipDataset(Dataset):
    def __init__(self, indices, model_name, train=False):
        self.indices = list(indices)
        self.is_satvit = model_name == 'satvit'
        self.img_size = MODELS[model_name]['img_size']
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        img = torch.from_numpy(images[idx].astype(np.float32))  # (4, 500, 500)

        if self.is_satvit:
            mean = img.mean(dim=(1, 2), keepdim=True)
            std = img.std(dim=(1, 2), keepdim=True)
            img = (img - mean) / (std + 1e-8)
        else:
            img = ((img - BAND_LO) / (BAND_HI - BAND_LO)).clamp(0, 1)
            img = (img[RGB_BANDS] - IMAGENET_MEAN) / IMAGENET_STD

        size = (self.img_size, self.img_size)
        img = F.interpolate(img[None], size=size, mode='bilinear', antialias=True)[0]

        if self.is_satvit:
            x = torch.zeros(SATVIT_CHANNELS, *size)
            x[SATVIT_BAND_SLOTS] = img
        else:
            x = img

        if self.train:
            if random.random() > 0.5:
                x = torch.flip(x, dims=[2])
            if random.random() > 0.5:
                x = torch.flip(x, dims=[1])
            x = torch.rot90(x, random.randint(0, 3), dims=[1, 2])

        return x, int(labels[idx]), names[idx]


def collate(batch):
    imgs, labels_, chip_names = zip(*batch)
    return torch.stack(imgs), torch.tensor(labels_), list(chip_names)


def make_loaders(model_name):
    batch_size = MODELS[model_name]['batch_size']
    loaders = []
    for idx, train in [(train_idx, True), (val_idx, False), (test_idx, False)]:
        loaders.append(
            DataLoader(
                WhaleChipDataset(idx, model_name, train=train),
                batch_size=batch_size,
                shuffle=train,
                num_workers=NUM_WORKERS,
                collate_fn=collate,
                pin_memory=True,
            )
        )
    return loaders

In [ ]:
# Sanity check: one training batch for the CNN/ViT input, shown as RGB.
imgs, batch_labels, batch_names = next(iter(make_loaders('efficientnet_b0')[0]))
print(imgs.shape, batch_labels.tolist())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, img, lab, name in zip(axes, imgs, batch_labels, batch_names):
    rgb = (img * IMAGENET_STD + IMAGENET_MEAN).clamp(0, 1).permute(1, 2, 0)
    ax.imshow(rgb)
    ax.set_title(f'{CLASSES[lab]} ({name})')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Models

EfficientNet-B0 and DeiT-Tiny come from `timm` exactly as in the 605 notebook, with 3 output classes
instead of 2.

For SatViT, the encoder, weight loading and freezing (only encoder layers 10 and 11 train) are Joe's
code, imported from `satvit/` unchanged. His flood decoder upsampled all 1024 patch encodings back
into a mask. For one label per chip, the head here averages the patch encodings and maps them to
class scores.

In [ ]:
# Joe's models.py imports `SatViT.SatViT_model`, which imports `transformer_model` directly.
sys.path.insert(0, str(SATVIT_DIR / 'SatViT'))
sys.path.insert(0, str(SATVIT_DIR))
from models import build_satvit_encoder  # noqa: E402


class SatViTClassifier(nn.Module):
    def __init__(self, num_classes, encoder_dim=768):
        super().__init__()
        self.encoder = build_satvit_encoder(device='cpu')
        self.encoder.load_state_dict(torch.load(SATVIT_WEIGHTS, map_location='cpu'))

        # Partial freezing from Joe's notebook: train only the last 2 encoder layers.
        for name, param in self.encoder.named_parameters():
            param.requires_grad = 'encoder.layers.10' in name or 'encoder.layers.11' in name

        self.head = nn.Sequential(nn.LayerNorm(encoder_dim), nn.Linear(encoder_dim, num_classes))

    def forward(self, x):
        patches = rearrange(x, 'b c (h i) (w j) -> b (h w) (c i j)', i=8, j=8)
        encodings = self.encoder.encode(images_patches=patches)  # (B, 1024, 768)
        return self.head(encodings.mean(dim=1))


def build_model(name: str) -> nn.Module:
    if name == 'satvit':
        model = SatViTClassifier(num_classes=len(CLASSES))
    else:
        model = timm.create_model(
            MODELS[name]['timm_name'], pretrained=True, num_classes=len(CLASSES)
        )
    return model.to(DEVICE)

## 7. Train / eval loops

`run_epoch` and `train_model` are the 605 versions with two changes:

- The loss is weighted by class. With 100 boats and 8 water chips, the model would otherwise learn
  to always say "boat".
- The optimizer only gets parameters that are set to train, which matters for SatViT's frozen layers.

`evaluate` reports per-class precision and recall and a confusion matrix, instead of the 605
per-generator breakdown.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    train_mode = optimizer is not None
    model.train(train_mode)
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    for imgs, labels_, _ in loader:
        imgs, labels_ = imgs.to(DEVICE, non_blocking=True), labels_.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(imgs)
            loss = criterion(logits, labels_)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels_.cpu().tolist())

    return total_loss / n, accuracy_score(all_labels, all_preds)


def train_model(name, train_loader, val_loader):
    print(f'\n{"=" * 60}\nTRAINING: {name}\n{"=" * 60}')
    model = build_model(name)

    counts = np.bincount(labels[train_idx], minlength=len(CLASSES))
    class_weights = torch.tensor(counts.sum() / (len(CLASSES) * counts), dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    best_val_loss = float('inf')
    best_state = None
    best_epoch = -1
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = run_epoch(model, val_loader, criterion, None)
        scheduler.step()
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)

        print(
            f'[{name}] epoch {epoch:02d}/{NUM_EPOCHS}  '
            f'train_loss={tr_loss:.4f} train_acc={tr_acc:.4f}  '
            f'val_loss={vl_loss:.4f} val_acc={vl_acc:.4f}  '
            f'({time.time() - t0:.0f}s)'
        )

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            epochs_no_improve = 0
            ckpt_path = os.path.join(CKPT_DIR, f'{name}_chip_best.pt')
            torch.save(
                {
                    'model_state_dict': best_state,
                    'epoch': epoch,
                    'val_loss': vl_loss,
                    'val_acc': vl_acc,
                    'model_name': name,
                    'img_size': MODELS[name]['img_size'],
                    'classes': CLASSES,
                },
                ckpt_path,
            )
            print(f'    ↳ new best, saved → {ckpt_path}')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'    ↳ early stopping at epoch {epoch} (best was {best_epoch}).')
                break

    model.load_state_dict(best_state)
    return model, history


def evaluate(model, loader, title):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels_, _ in loader:
            preds = model(imgs.to(DEVICE, non_blocking=True)).argmax(1).cpu().tolist()
            all_preds.extend(preds)
            all_labels.extend(labels_.tolist())

    print(f'\n── {title} ──')
    print(
        classification_report(
            all_labels,
            all_preds,
            labels=list(range(len(CLASSES))),
            target_names=CLASSES,
            zero_division=0,
        )
    )
    ConfusionMatrixDisplay.from_predictions(
        all_labels, all_preds, labels=list(range(len(CLASSES))), display_labels=CLASSES
    )
    plt.title(title)
    plt.show()
    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro', zero_division=0),
    }


def plot_history(history, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history['train_loss'], label='Training Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel('Loss (weighted cross-entropy)')
    plt.legend()
    plt.show()

## 8. Train each model

In [ ]:
results = {}


def run(name):
    train_loader, val_loader, test_loader = make_loaders(name)
    model, history = train_model(name, train_loader, val_loader)
    plot_history(history, f'{name} learning curve')
    results[name] = evaluate(model, test_loader, f'{name} — test')
    # Free VRAM before the next model
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
run('efficientnet_b0')

In [ ]:
run('deit_tiny')

In [ ]:
if SATVIT_WEIGHTS.exists():
    run('satvit')
else:
    print(f'Skipping SatViT: put SatViT_V2.pt at {SATVIT_WEIGHTS} (link in satvit/README.md).')

## 9. Compare

Macro F1 averages the three classes equally, so it shows whether the rare classes (water, whale)
are being found. Plain accuracy mostly reflects boats.

In [ ]:
print(f'{"model":18s} {"accuracy":>9s} {"macro F1":>9s}')
for name, r in results.items():
    print(f'{name:18s} {r["accuracy"]:9.3f} {r["macro_f1"]:9.3f}')